In [1]:
#First install libraries
import pyodbc
import pandas as pd 
import warnings
import numpy as np 
from scipy import stats
from scipy.stats import chi2_contingency
warnings.filterwarnings("ignore")

In [2]:
#set connection

import os
import pyodbc

required_env_vars = {
    "BANK_CHURN_DB_SERVER": os.getenv("BANK_CHURN_DB_SERVER"),
    "BANK_CHURN_DB_NAME": os.getenv("BANK_CHURN_DB_NAME"),
    "BANK_CHURN_DB_USER": os.getenv("BANK_CHURN_DB_USER"),
    "BANK_CHURN_DB_PASSWORD": os.getenv("BANK_CHURN_DB_PASSWORD"),
}

missing_env_vars = [key for key, value in required_env_vars.items() if not value]
if missing_env_vars:
    raise EnvironmentError(
        "Missing required database environment variables: " + ", ".join(missing_env_vars)
    )

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    f"Server={required_env_vars['BANK_CHURN_DB_SERVER']};"
    f"Database={required_env_vars['BANK_CHURN_DB_NAME']};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
    f"UID={required_env_vars['BANK_CHURN_DB_USER']};"
    f"PWD={required_env_vars['BANK_CHURN_DB_PASSWORD']};"
)

cursor = conn.cursor()
cursor.execute("SELECT DB_NAME();")
print(cursor.fetchone())






('BankChurn',)




Before building a churn prediction model , we need to check the relationships between variables using the three statistical tests. This step is important because it will help us identify which variables actually influence customer churn.




**First Test: Independent Samples t-Test**

Business question:

In banking, account balance is a key indicator of customer value. The business wants to answer:

**Do customers who churn have different account balances than loyal customers?**

This question directly informs strategic decisions such as:

1. Retention campaign targeting  
2. VIP customer protection  
3. Risk segmentation  
4. Revenue forecasting  

Interpretation:

- If high-balance customers are more likely to churn, the bank faces higher risk from losing high-value clients and should prioritize immediate retention and protection actions.  
- If low-balance customers churn more often, this may signal issues with engagement, product fit, or overall value proposition.


**Statistical Hypotheses**

We compare the average account balance between two groups:

1. Customers who churned  
2. Customers who were retained (loyal)

**Null hypothesis (H0):** The mean account balance is the same for both groups.  
**Alternative hypothesis (H1):** The mean account balance is different between the two groups.

This is a **two-tailed** test because we are checking for any difference, not a specific direction.



**Column Selection**

To perform thys hypothesis test, only two columns are required:

1. Balance represents the continuous numerical measures being compared 
2. Churned defines the two customers groups (churned vs retained) used for statistical comparison

**Test Assumptions**

The independent Sample t-test relies on several important assumptions:

1. **Independence**: Observations must be independent of one another.In banking datasets, each customer represents a unique record, so this assumption is satisfied 

2. **Normality**:The distribution of Balance within each group should be approximately normal. If the sample size is large (n> 30 per group), the t-test is considered relatively robust to normality violations due to the central limit Theorem

3. **Homogeneity of Variance**: The variance of Balance should be equal across the two groups. This assumption can be tested using Levene's test.If the variance are unequal , the appropiate alternative is Welch's t-test



**Step 1 --Data Preparation**

In [3]:
query = """
SELECT
    a.Balance,
    d.Churned
FROM account a
JOIN demographic d
    ON a.CustomerId = d.CustomerId
"""

df_t_test = pd.read_sql(query, conn)
df_t_test.head()


,Balance,Churned
0,119827.49,True
1,83807.86,False
2,159660.80,True
3,119827.49,False
4,125510.82,False


**Step 2-- Split Groups**

In [4]:
churned_t_test = df_t_test[df_t_test["Churned"] == 1 ]["Balance"]
retained_t_test =df_t_test[df_t_test["Churned"] == 0 ]["Balance"]

**Step 3--Descriptive Stats**


In [5]:
print("Churned Mean:", churned_t_test.mean())
print("Retained Mean:",retained_t_test.mean())

Churned Mean: 120521.27620520373
Retained Mean: 119650.01690066556


**Step 4-- Variance Check**

In [6]:
# if p-value > 0.05, equal variances assumed
stats.levene(churned_t_test,retained_t_test)


LeveneResult(statistic=np.float64(36.69675448689501), pvalue=np.float64(1.4298234841380065e-09))

Since p-value is less than 0.05, H0 is rejected.Therefore, variances are not equal.


**Step 5-- Run t-test**

Since variances are not equal , we should use **Welch's t-test**


In [7]:
t_stat, p_value = stats.ttest_ind(
    churned_t_test,
    retained_t_test,
    equal_var=False
)

print("t-stat:", t_stat)
print("p-value:", p_value)


t-stat: 1.3534724523393942
p-value: 0.17601045694361087


After carrying this test , we found out p-value is more than 0.05. This means we fail to reject null hyptohesis. Then , **there is no difference between balance amount for churned and retaiend people**. It shows that churining of clients is not caused by amount of balance account. Therefore , account balance may not be a strong standalone predictor of churn and may provide limited explanatory power in the predictive model 

**Test 2: Chi-Square Test of Independence**

**Business question**

In banking, account activity (customer engagement) is often a strong signal of churn risk. The business wants to understand:

**Are inactive customers more likely to churn?**

If this dependency is true, the insight can directly support several strategic initiatives:

1. Early warning system development  
2. Reactivation campaign design  
3. Customer health score modeling  

If customer activity is significantly associated with churn:

- Engagement becomes a key driver of churn.  
- Behavior-based retention strategies are required to reduce attrition risk.



**Statistical Hypotheses**

We are now examining the relationship between two categorical variables:

1. `Churned` -> Yes/No  
2. `IsActive` -> Active/Inactive  

- **Null Hypothesis (H0):** Churn status and customer activity are independent.  
- **Alternative Hypothesis (H1):** Churn status and customer activity are dependent.

To perform this hypothesis test, we only need these two columns:

- `IsActive` indicates whether the account is still active.  
- `Churned` defines the two customer groups (churned vs retained) used for statistical comparison.



**Contingency table Concept :**

The chi-square test uses a frequency table (contingency table).
It checks whether the observed counts across categories are significantly different from the counts we would expect if the two variables were independent (unrelated).




**Test Assumptions:**

To ensure vailidity of the Chi-square test ,the following assumptions must be satisfied:

-Variables are categorical: both variables must be categorical
-Independence of observations: Each observation must be independent. In the banking dataset , each customer represents a unique record , so the independence assumptions holds.
-Expected frequency >5 : Each cell in the contigency table should have an expected frequency of at least 5. If this condition is not met , we use Exact Test. 


**Why this test?**
The reason why we choose the statistical test is based on the structure of the variables and the analytical objectives:

-Dependent variable is categorical 
-Independent variable is binary categorical
-Objective is association

**Implement Test**

**Step 1 -- Data Preparation**


In [8]:

query = """
SELECT
    d.Churned,
    a.IsActive
FROM demographic d
JOIN account a
    ON d.CustomerId = a.CustomerId
"""

df_chi = pd.read_sql(query, conn)
df_chi.head()


,Churned,IsActive
0,True,True
1,False,True
2,True,False
3,False,False
4,False,True


Step 2--**Build contingency Table**


In [9]:
ct= pd.crosstab(df_chi["Churned"],df_chi["IsActive"])
print(ct)

IsActive  False  True 
Churned               
False      3547   4416
True       1302    735


Step 3 -- **Run Chi-square test**


In [10]:
chi2,p_value,dof,expected = chi2_contingency(ct)

print("chi2:",chi2)
print("p_value",p_value)
print ("Degrees of freedom:",dof)

chi2: 242.98534164287963
p_value 8.785858269303703e-55
Degrees of freedom: 1


In [11]:
print(expected)

[[3861.2587 4101.7413]
 [ 987.7413 1049.2587]]


**Output Interpretation**
The p-value is less than 0.05. This means we reject the null hypotheses of test.So , finally interpretation is that churn and customer activity are associated

**Test 3: ANOVA TEST**

**Business Question:**

The bank operates across multiple countries/regions. Business wants to understand :

`Does customer tenure differ significantly across countries?`

Why is this question important? 

If tenure is lower in certain countries, it may indicate that :

-Customers in those markets tend to leave earlier or maintain shorter relationships with the bank 
-Service quality, competition, product offerings or digital experience may be weaker in those regions
-Customer acquisition and attrition dynamics differ by market 

This results could bring us ideas on :

-Designing country-specific retention strategies 
-Increasing investment in products or channels in markets with lower tenure
-Benchmarking regional and branch performance across countries


**Statistical hypotheses:**
-Null Hyptohesis: The average of tenure is all countries are the same
-Alternative hypothesis:The average of tenure is all countries are not the same


**Column Selection**

To perform this hypothesis test, only two columns are required :

-Tenure as the numerical column shows how long the client has been using services 
-Geography as the multiple categorical column shows different locations 


**Assumption Checks (One-Way ANOVA)**

Before running ANOVA, we verify:

1. **Sample size per group**  
  Each country should have an adequate number of observations

2. **Independent observations**  
   - Each customer appears once, and observations are not duplicated across groups.

3. **Approximate normality within each group**  
   - `Tenure` should be approximately normally distributed inside each `Geography` group.

4. **Homogeneity of variances**  
    - If group variances are similar, use **standard one-way ANOVA**.  
    - If variances are unequal, use **Welch’s ANOVA** 
So, when homogeneity of variance is not met, we proceed with **Welch’s ANOVA** instead of classical ANOVA.


WHY THIS TEST ?

The choice of statistical test is based on the structure of the variables and the analytical objective

-Dependent variable is numerical (Tenure)
-Independent variable is multi-categorical (Location)
-Objective is average mean





**Implement test**



**Step 1 -- Data preparation**



In [12]:

query = """
SELECT
    l.Geography AS Country,
    a.Tenure
FROM account AS a
JOIN demographic AS d
    ON a.CustomerId = d.CustomerId
JOIN Location AS l
    ON d.LocationId = l.LocationId
WHERE a.Tenure IS NOT NULL
  AND l.Geography IS NOT NULL
ORDER BY a.CustomerId;
"""

df_anova = pd.read_sql(query, conn)
df_anova.head()



,Country,Tenure
0,Spain,9
1,Germany,1
2,USA,1
3,Spain,6
4,Spain,10


**Step 2 -- Descriptive stats**


In [13]:
summary = df_anova.groupby("Country")["Tenure"].agg(["count","mean","std"])
print(summary.sort_values("mean",ascending=False))


         count      mean       std
Country                           
UK        1703  5.127422  2.939195
France    1714  5.042590  2.904582
USA       1669  5.002397  2.890694
Germany   1626  4.987700  2.870085
Canada    1679  4.982132  2.888003
Spain     1609  4.927906  2.857046


**Step 3 -- Check Variance equality**


In [14]:
#if p>0.05, equal variances assumed

groups=[g["Tenure"].values for _,g in df_anova.groupby("Country")]
lev= stats.levene(*groups)
print(lev)

LeveneResult(statistic=np.float64(0.5338267546333629), pvalue=np.float64(0.7508283480014935))


Since p-value is more than 0.05, H0 is rejected. So variances are not equal.

Step 4-- Run test

In [15]:
f_stat,p_value_anova = stats.f_oneway(*groups)
print("F:",f_stat)
print("p_value:",p_value_anova)

F: 0.9151410065439708
p_value: 0.4698663379103213


**OUTPUT INTERPRETATION**

The p-value is greater than 0.05. It means we fail to reject null hypothesis of test. So, final interpretation is that `The Average of tenure is all countries are the same`
 

TEST 4: EFFECT SIZE 

So far in our results we found that :

Welch t-test:No significant balance mean difference between churned and retained customer
Chi-square:There is significant association between active account and churned customer
ANOVA TEST:No significant tenure mean difference between countries


Here, we would like to find how big is the impact:
| Cohen's d | Effect Size Label |
|---:|---|
| 0.20 | Very Small |
| 0.50 | Small |
| 0.8 | Medium |
| 1.20 + | Very Large |




In [16]:
# means
churned_mean = churned_t_test.mean()
retained_mean = retained_t_test.mean()

# std devs (sample std)
churned_std = churned_t_test.std(ddof=1)
retained_std = retained_t_test.std(ddof=1)

# sample sizes
churned_n = churned_t_test.shape[0]
retained_n = retained_t_test.shape[0]

# pooled standard deviation
pooled_std = np.sqrt(
    (((churned_n - 1) * churned_std**2) + ((retained_n - 1) * retained_std**2))
    / (churned_n + retained_n - 2)
)

# Cohen's d
cohens_d = (churned_mean - retained_mean) / pooled_std
print("Cohen's d:", cohens_d)





Cohen's d: 0.03623904081079434


The Independent Sample t-test indicated no statiscally significant difference in average account balance between churned and retained customers(high p-value)
Also , the effect size (Cohen's D = 0.03) is neglibible, suggesting that the balance has minimal practical impact on churn behavior.

| Metric | Intepretation |
|---:|---|
| High p-value is t-test  | The difference balance between churn and retain its not important |
| Small Cohen | Even if there is a significant difference, from business perspective, it was negligible |



**Cramer´s V : CHI-SQUARE**

The table below shows the results for interpretation:
| Cramer´s V | Effect Size |
|---:|---|
| 0.1  | Weak |
| 0.3 | Moderate |
| 0.5 | Strong |


In [17]:
n_chi=ct.sum().sum()
k_chi=min(ct.shape)

cramers_v=np.sqrt(chi2 / (n_chi * (k_chi - 1)))

print("Cramer´s v:", cramers_v)

Cramer´s v: 0.15587987094005423


The Chi-square test shows a statistically significant relationship between customer activity and churn . However `Cramer's V = 0.15` indicates a weak association suggesting that while engagement is related to churn , it is not the sole driving factor 

**Conclusion for Cramer´s V : CHI-SQUARE**


| Metric | Interpretation |
|---:|---|
| Small p-value is Chi-square test  | There is a statistically significant relationship between account activity status and customer churn. |
| Moderate Cramer | The association exists, but its strength is modest, which suggests activity is only one of several factors influencing churn.|



**ETA/OMEGA: ANOVA**

This table below shows the result for interpretation 

| OMEGA/ETA | EFFECT SIZE |
|---:|---|
| 0.01  | Small |
| 0.06 | Medium |
| 0.14 | Large  |




In [21]:
import numpy as np

# Calculate SS
all_values = df_anova["Tenure"].values
grand_mean = np.mean(all_values)

ss_between= sum(
    len(group) * (np.mean(group) - grand_mean)**2
    for group in groups
)

ss_within=sum(
    sum((group - np.mean(group))**2)
    for group in groups
)

ss_total = ss_between + ss_within
print (ss_total)

83638.36159999996


In [24]:
#CALCULATE ETA SCORE
K = len(groups)
N = len(all_values)

df_between = K - 1
df_within = N - K

ms_within = ss_within / df_within

eta_squared = ss_between / ss_total
print("Eta Squared(η2):", eta_squared)



Eta Squared(η2): 0.00045763568409215794


In [25]:
#Calculate Omega Score 
omega_squared = (
    ss_between - (df_between * ms_within)
) / (ss_total * ms_within)
print("Omega Squared (w²):",omega_squared)



Omega Squared (w²): -5.0729708134325995e-06


**CONLUSION:**

The one-way ANOVA test shows that there is no significant difference in the average customer tenure between countries because the p-value is greater than 0.05.

The effect size results support this conclusion. Both Eta Squared (0.00046) and Omega Squared (0) are extremely small, which means the practical impact is almost zero.

In simple terms, this means that the country a customer is from does not meaningfully affect how long they stay as a customer in this dataset.